#  ToolRetryMiddleware中间件
## 举例1，带抖动

In [3]:
from datetime import time, datetime

from langchain_core import messages
from langchain.agents.middleware import SummarizationMiddleware, LLMToolSelectorMiddleware, ToolRetryMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from typing import Union, Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from rich import print as rprint
from utils.tool_utils import query_weather_from_web,query_news_from_web

# 1.初始化模型
model= ChatOpenAI(
    model="qwen3-vl:latest",
    # model="483025889/qwen3.5:9b",
    # model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

# 2定义函数
def write_time(s:str):
    with open("call_times_with_jitter.txt",'a',encoding='utf-8') as f:
        f.write(s+'\n')

count = 1
start_time = None
# 3.定义工具
@tool
def get_weather(city:str):
    """查询指定城市天气"""
    global count
    global start_time
    interval = 0
    curr_time = datetime.now()
    if start_time is None:
        interval = 0
    else:
         # 计算当前调用与上一次调用之间的时间差（秒）
        interval = (curr_time - start_time).total_seconds()
    start_time = curr_time
    res_str = f"第 {count} 次调用，当前时间： {start_time}, 和上次调用间隔 {interval} 秒"
    count+=1
    # 写日志
    write_time(res_str)
     # 故意抛出 TimeoutError，以此触发中间件的重试机制
    raise TimeoutError("Not Implemented")

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        ToolRetryMiddleware(
            max_retries=6,# 最大重试次数（不包含初始的那次调用，一共最多调 1 + 6 = 7 次）
            backoff_factor=2.0,# 指数退避因子（每次重试等待时间乘以 2）
            initial_delay=1.0, # 第一次重试前的初始等待时间（1 秒）
            max_delay=10.0, # 最大等待延迟上限（防止指数增长无限大，限制在 10 秒）
            jitter=True, # 开启抖动（在等待时间中加入随机性，防止并发请求时出现“惊群效应”）
            retry_on=(TimeoutError,),# 仅针对捕获到特定的 TimeoutError 异常时才触发重试
            on_failure="continue" # 当达到最大重试次数依然失败时，Agent 的行为："continue" 表示将错误信息包装后塞回对话历史，让大模型知道失败了并继续决策
        )
    ]
)

resp = agent.invoke({"messages":[HumanMessage("北京今天的天气如何？")]})

for msg in resp["messages"]:
    msg.pretty_print()



================================ Human Message =================================

北京今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_9ydhsowe)
 Call ID: call_9ydhsowe
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

Tool 'get_weather' failed after 7 attempts with TimeoutError: Not Implemented. Please try again.
================================== Ai Message ==================================

很抱歉，目前无法获取北京的天气信息，系统出现了技术故障（TimeoutError）。建议稍后再尝试查询，或检查网络连接是否正常。如果您有其他需求，我很乐意为您提供帮助。


## 举例2.无抖动

In [5]:
from datetime import time, datetime

from langchain_core import messages
from langchain.agents.middleware import SummarizationMiddleware, LLMToolSelectorMiddleware, ToolRetryMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from typing import Union, Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from rich import print as rprint
from utils.tool_utils import query_weather_from_web,query_news_from_web

# 1.初始化模型
model= ChatOpenAI(
    model="qwen3-vl:latest",
    # model="483025889/qwen3.5:9b",
    # model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

# 2定义函数
def write_time(s:str):
    with open("call_times_with_jitter.txt",'a',encoding='utf-8') as f:
        f.write(s+'\n')

count = 1
start_time = None
# 3.定义工具
@tool
def get_weather(city:str):
    """查询指定城市天气"""
    global count
    global start_time
    interval = 0
    curr_time = datetime.now()
    if start_time is None:
        interval = 0
    else:
         # 计算当前调用与上一次调用之间的时间差（秒）
        interval = (curr_time - start_time).total_seconds()
    start_time = curr_time
    res_str = f"第 {count} 次调用，当前时间： {start_time}, 和上次调用间隔 {interval} 秒"
    count+=1
    # 写日志
    write_time(res_str)
     # 故意抛出 TimeoutError，以此触发中间件的重试机制
    raise TimeoutError("Not Implemented")

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        ToolRetryMiddleware(
            max_retries=6,# 最大重试次数（不包含初始的那次调用，一共最多调 1 + 6 = 7 次）
            backoff_factor=2.0,# 指数退避因子（每次重试等待时间乘以 2）
            initial_delay=1.0, # 第一次重试前的初始等待时间（1 秒）
            max_delay=10.0, # 最大等待延迟上限（防止指数增长无限大，限制在 10 秒）
            jitter=False, # 关闭抖动，意味着重试机制从“随机化的指数退避”退化成了“严格固定的指数退避
            retry_on=(TimeoutError,),# 仅针对捕获到特定的 TimeoutError 异常时才触发重试
            on_failure="continue" # 当达到最大重试次数依然失败时，Agent 的行为："continue" 表示将错误信息包装后塞回对话历史，让大模型知道失败了并继续决策
        )
    ]
)

resp = agent.invoke({"messages":[HumanMessage("北京今天的天气如何？")]})

# 1. 你的提问 -> 2. AI 决定调用工具 -> 3. 重试失败后的错误反馈 -> 4. AI 最终给出的兜底回复
for msg in resp["messages"]:
    msg.pretty_print()



================================ Human Message =================================

北京今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_wgyxgbok)
 Call ID: call_wgyxgbok
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

Tool 'get_weather' failed after 7 attempts with TimeoutError: Not Implemented. Please try again.
================================== Ai Message ==================================

很抱歉，当前无法获取北京的实时天气信息（系统提示服务暂时不可用）。建议您稍后再尝试查询，或检查网络连接是否正常。需要我为您做其他协助吗？
